In [1]:
print(123)

123


In [2]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-08-05 11:35:42--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.1’

rag_helper.py.1     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-08-05 11:35:42 (29.6 MB/s) - ‘rag_helper.py.1’ saved [2134/2134]

--2026-08-05 11:35:42--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 20

In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [4]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [5]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [6]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download for your operating system:
   - macOS: download and install the `.pkg`
   - Windows: download and install the `.msi`
   - Linux: run
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. After installation, open a terminal and run:
   ```bash
   ollama run llama3
   ```

   This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

3. To test that the local Ollama server is running, you can use:
   ```bash
   curl http://localhost:11434
   ```

   You should get a response like:
   ```json
   {"models": [...]}
   ```

If you want to use it from Python, install the client with:

```bash
pip install ollama
```

and then use:

```python
import ollama

response = ollama.chat(
    model='llama3',
    messages=[{"role": "user", "content": your_prompt}]
)

print(response['message']['content'])
```


In [7]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

You can run the course locally if you’re comfortable setting up the needed tools, but the FAQ context does not mention “Olama” specifically.

For local setup, the course says you’d need to set up:
- Python
- `uv`
- Jupyter
- Docker
- any other tools needed for the module

If you want, I can also help you interpret whether you meant a specific local tool from the course context.


In [8]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Usually, yes — if the course is still open and you meet any requirements.\n\nA few things to check:\n- **Enrollment status:** Is registration still open?\n- **Prerequisites:** Do you need prior experience or another course first?\n- **Capacity:** Some courses have limited spots.\n- **Schedule/format:** Make sure it fits your availability.\n\nIf you want, I can help you figure out **whether you’re eligible** if you share the course name or the course rules.'

In [17]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [10]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [11]:
len(response.output)

1

In [12]:
call = response.output[0]

In [13]:
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered just discovered can I join enrollment late registration how to join"}', call_id='call_udhObRh7m9w5aECYeVYBSGX0', name='search', type='function_call', id='fc_0d08cad138e0dc98006a732042cda081a1b50b6ae6eb941fdd', caller=None, namespace=None, status='completed')

In [14]:
#We need to pass the arguments
import json

args = json.loads(call.arguments)
args

{'query': 'join course discovered just discovered can I join enrollment late registration how to join'}

In [15]:
#this is the name of the tool that llm used (search)
call.name

'search'

In [18]:
#we keep the results in a variable, and this is what llm will use (we pass it to it)
results = search(**args)

In [19]:
result_json = json.dumps(results, indent=2)

In [20]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [21]:
messages.append(call)

In [22]:
messages.append(function_call_output)

In [23]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered just discovered can I join enrollment late registration how to join"}', call_id='call_udhObRh7m9w5aECYeVYBSGX0', name='search', type='function_call', id='fc_0d08cad138e0dc98006a732042cda081a1b50b6ae6eb941fdd', caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_udhObRh7m9w5aECYeVYBSGX0',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LL

In [24]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [25]:
print(response.output_text)

Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while the course is still accepting submissions.


In [26]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(979, 35)

In [27]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(652, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0001176


The following code corresponds to the section "The agentic loop".

In [28]:
#converts json string into python dicctionary
#invoke the specific function that llm decides (currently we only have search)
#this is a helper function.

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [29]:
#The current set up for the agent loop: instructions, question, messages, and we send this to the LLM. 
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [30]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [31]:
#So in the response we obtain several results (corresponding to subsequent iterations)
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        #process function call
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        #do something else
        print('ASSISTANT:')
        print(item.content[0].text)

function_call: search {"query":"join the course enroll registration discovered course can I join"}


In [32]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join the course enroll registration discovered course can I join"}', call_id='call_XYEfwAXBmqL8HFitoCaehCnb', name='search', type='function_call', id='fc_0dcff1170299b422006a732084e04481a19ef99c650a469c3e', caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_XYEfwAXBmqL8HFitoCaehCnb',
  'output': '[\n  {\n    "id"

In [33]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

#here we want to check if there are more function calls (to exist the loop)
it = 1
#we loop forever until function_calls = FALSE, and then we brake the loop. 
while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break


iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment"}
function_call: search {"query":"course FAQ join late enrollment discovered course"}
function_call: search {"query":"can I still enroll in the course if I just discovered it"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If your goal is a certificate, the key thing is to submit your project while submissions are still open. You can still work through the material even in self-paced mode, but the certificate requires completing the capstone/project during a live cohort window.

If you want, I can also help you with:
- how to start the course,
- whether you can still get a certificate,
- or the weekly workflow.


In [34]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [35]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [36]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join the course enrollment discovered course can I join late enrollment access FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you’ll need to submit your project while submissions are still being accepted by the live cohort.

If you just want to learn, you can start anytime and work through the materials at your own pace.

Would you like me to help you with the next steps for getting started?


In [37]:
result

'Yes — you can still join the course.\n\nIf you want a certificate, though, you’ll need to submit your project while submissions are still being accepted by the live cohort.\n\nIf you just want to learn, you can start anytime and work through the materials at your own pace.\n\nWould you like me to help you with the next steps for getting started?'

In [38]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit queen's gambit chess opening"}
iteration #2...
function_call: search {"query":"queen gambit"}
iteration #3...
ASSISTANT:
I couldn’t find any course FAQ entry related to “queen gambit,” so it looks like this is off-topic for the course.

If you meant something specific from the course materials or logistics, feel free to rephrase it, and I can check again. Are there other areas that you want to explore?


This last part of the code corresponds to the use of frameworks
Here we are using toyakit to illustrate the use of frameworks. We can try later other more production-based ones, like LangGraph.

In [39]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [40]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [41]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'}
    )

In [42]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [43]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [44]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [45]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [46]:
result = runner.loop(
    prompt='How do I run Olama locally?',
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [47]:
result.cost

CostInfo(input_cost=Decimal('0.0029985'), output_cost=Decimal('0.0015255'), total_cost=Decimal('0.0045240'))

In [48]:
result.all_messages

[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searchers. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run local install FAQ"}', call_id='call_64A

In [49]:
result2 = runner.loop(
    prompt='How do I run a different model?',
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [50]:
runner.run();

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


KeyboardInterrupt: Interrupted by user